# Clean up ShareChat

- Dataset - https://huggingface.co/datasets/tucnguyen/ShareChat/tree/main
- Paper - https://arxiv.org/abs/2512.17843

In [1]:
import pandas as pd

In [15]:
# takes a few min
# Let's get: 1st turn only; English language; strings <= 131 characters (same filtering as WildChat)

dfs = []
models = ['claude', 'grok', 'perplexity', 'gemini', 'chatgpt']

for model in models:
    dfs.append(
        pd.read_csv(
            f'sharechat/{model}_results_final_language_filtered.csv'
        ).query(
            'detected_language_final == "English"'
        ).query(
            'role == "user"'
        ).query(
            'message_index == 1' # Let's look at the first message from the user only
        ).assign(
            first_turn=lambda df_: df_.plain_text.str.strip(),
            first_turn_len=lambda df_: df_.first_turn.str.len(),
        ).query(
            'first_turn_len <= 131'
        )
        .drop_duplicates(subset=['first_turn', 'platform'])
        .assign(
            conv_id=lambda df_: 'sharechat::' + df_.url
        )
    )
    
data = pd.concat(dfs)

In [17]:
data

,Unnamed: 0,platform,url,turns_count,message_index,role,plain_text,model,created_at,published_at,...,message_create_time,thinking,links,source,last_updated,source_bar,views,shares,other_info,create_time
0,0,claude,https://gemini.google.com/share/f3f610bcf418,2,1,user,Is resetting cmos and flashing the bios the same?,2.5 Flash,"July 23, 2025 02:43 AM","July 23, 2025 02:45 AM",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
32,32,claude,https://gemini.google.com/share/48febbdabcbb,1,1,user,Craft song lyrics entitled Nagas,Gemini,"February 28, 2025 08:36 PM","February 28, 2025 08:40 PM",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
580,580,claude,https://gemini.google.com/share/9196c2995124,1,1,user,Did the <REDACTED> spend a significant amount ...,Gemini,"May 15, 2025 12:38 PM","May 15, 2025 12:38 PM",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
584,584,claude,https://gemini.google.com/share/bfbda5ddbff7,1,1,user,Is black Friday the day after tomorrow?,Gemini Advanced,"November 27, 2024 08:50 PM","November 27, 2024 08:51 PM",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
652,652,claude,https://gemini.google.com/share/22e417c32fce,1,1,user,Top issues for <REDACTED> judge vote?,Gemini,"April 2, 2025 01:05 PM","April 2, 2025 01:06 PM",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
976308,1062843,chatgpt,https://chatgpt.com/share/67cdc750-5ba8-8003-a...,1,1,user,If <REDACTED> violates the agreement or treaty...,default_model_slug,NaN,NaN,...,ts:1741539109.309220,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-03-09 12:52:32
976324,1062859,chatgpt,https://chatgpt.com/share/67fd5986-9a68-8012-a...,5,1,user,Graph the volume and cost per incident of prop...,default_model_slug,NaN,NaN,...,ts:1744655211.856000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-04-14 14:52:54
976334,1062869,chatgpt,https://chatgpt.com/share/6764c2e6-2478-8012-9...,2,1,user,One of the biggest things that I have noticed,default_model_slug,NaN,NaN,...,ts:1734656710.154401,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-12-19 20:05:42
976338,1062909,chatgpt,https://chatgpt.com/share/e2fc5cba-40f8-40ea-8...,1,1,user,<DATE_TIME>/FT Quadrant: Horizon Event Ship - ...,text-davinci-002-render-sha,NaN,NaN,...,ts:1686505630.827332,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-06-11 13:52:50


In [20]:
data.filter(['conv_id', 'first_turn']).to_csv('interim/sharechat.first_turn.en.len131.csv.zip')